# 🧠 03 — Thinking Out Loud: Chain-of-Thought for Driving Safety

> **Goal:** turn on "show your work" mode so the agent **reasons step by step** before
> answering — just like a careful driving instructor.

---

## What is Chain-of-Thought?

When you solve a hard problem, you don't blurt the answer — you think through it. Setting
**`reasoning=True`** makes Cosmos do the same: it writes its private thinking inside a
`<think>...</think>` block, *then* gives the final answer.

```mermaid
flowchart LR
    VID["dashcam clip"]:::data
    M["CosmosVisionModel<br/>reasoning=True"]:::reason
    T["&lt;think&gt;<br/>spots the cyclist,<br/>checks distances,<br/>weighs risk…<br/>&lt;/think&gt;"]:::think
    ANS([Final safety advice]):::out
    VID --> M --> T --> ANS
    classDef user fill:#ECEFF1,stroke:#607D8B,stroke-width:2px,color:#263238
    classDef agent fill:#E3F2FD,stroke:#1976D2,stroke-width:2px,color:#0D47A1
    classDef reason fill:#E8F5E9,stroke:#388E3C,stroke-width:2px,color:#1B5E20
    classDef gen fill:#F3E5F5,stroke:#8E24AA,stroke-width:2px,color:#4A148C
    classDef tool fill:#FFF3E0,stroke:#F57C00,stroke-width:2px,color:#E65100
    classDef data fill:#FFFDE7,stroke:#FBC02D,stroke-width:2px,color:#F57F17
    classDef out fill:#E0F2F1,stroke:#00897B,stroke-width:2px,color:#004D40
    classDef think fill:#FCE4EC,stroke:#D81B60,stroke-width:2px,color:#880E4F
```

🧠 **Pink = the thinking step.** It's like seeing the scratch paper, not just the answer.

In [ ]:
# 🔧 Preflight — this cell never crashes. It just tells us what's available.
import os, sys, time
os.environ.setdefault("QWEN_VL_VIDEO_READER", "torchcodec")
sys.path.insert(0, os.path.abspath(".."))

# 🔒 Cosmos tools confine file access to a workspace allow-list for safety.
# The notebooks live in notebooks/ but our sample media sits one level up, so we
# explicitly widen the workspace to the project root + /tmp. (This is how you grant
# the tools access to a folder — never wider than you need.)
os.environ["COSMOS_WORKSPACE"] = os.pathsep.join([os.path.abspath(".."), "/tmp"])            # import strands_cosmos from repo root

PROJECT_ROOT = os.path.abspath("..")
SAMPLE_VIDEO = os.path.join(PROJECT_ROOT, "sample.mp4")
SAMPLE_IMAGE = os.path.join(PROJECT_ROOT, "sample.png")

def gpu_ready() -> bool:
    try:
        import torch
        return torch.cuda.is_available()
    except Exception:
        return False

READY = gpu_ready()
print("🎮 GPU available :", READY)
if READY:
    import torch
    print("   Device       :", torch.cuda.get_device_name(0))
    print(f"   VRAM         : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
else:
    print("   (CPU-only is fine for reading along — the heavy cells will skip safely.)")
print("📹 sample.mp4   :", os.path.exists(SAMPLE_VIDEO))
print("🖼  sample.png   :", os.path.exists(SAMPLE_IMAGE))

## Step 1 — Load the model with thinking switched ON
The only new thing here is `reasoning=True`. Everything else is the same vision model from
notebook 02.

In [ ]:
agent = None
if READY and os.path.exists(SAMPLE_VIDEO):
    from strands import Agent
    from strands_cosmos import CosmosVisionModel
    t0 = time.time()
    model = CosmosVisionModel(
        model_id="nvidia/Cosmos-Reason2-2B",
        reasoning=True,
        fps=4,
        params={"max_tokens": 2048, "temperature": 0.6},
    )
    agent = Agent(model=model)
    print(f"✅ Chain-of-thought agent ready in {time.time()-t0:.1f}s")
else:
    print("⏭  Need a GPU + sample.mp4.")

## Step 2 — Ask a safety question
We give it a real driving-instructor task: find the hazards and recommend actions.
Watch for the `<think>` block in the output.

In [ ]:
if agent is not None:
    t0 = time.time()
    agent(
        f"<video>{SAMPLE_VIDEO}</video> "
        "Analyze this dashcam footage for safety hazards. Identify: "
        "1) moving objects, 2) collision risks, 3) recommended driver actions."
    )
    print(f"\n⏱  {time.time()-t0:.1f}s")
else:
    print("⏭  Skipped.")

## Step 3 — Put events on a timeline
Chain-of-thought is also great for ordering things in time.

In [ ]:
if agent is not None:
    agent(f"<video>{SAMPLE_VIDEO}</video> Describe the notable events in chronological order.")
else:
    print("⏭  Skipped.")

# 🎓 What you learned

| Concept | Takeaway |
|---------|----------|
| `reasoning=True` | Turns on step-by-step **chain-of-thought** |
| `<think>…</think>` | The model's visible "scratch paper" |
| When to use it | Safety, planning, multi-step or high-stakes questions |

!!! tip "Trade-off"
    Thinking takes more tokens (slower, longer output). Use it when **correctness matters
    more than speed**.

**Next:** [04 — Embodied Reasoning](04_embodied_reasoning.ipynb) — what should a **robot**
do next? 🤖 →